# 01 — Denoise + VAD baseline

Tests the two audio front-end stages per CLAUDE.md's model stack.
Unlike ASR/MT/TTS, these were implemented directly in Rust first
(`src/stages/denoise.rs`, `src/stages/vad.rs` — see
`docs/notes/NOTE.md`'s 2026-07-15 entries) since both are simple enough
(single forward pass per audio frame, no autoregressive decode) not to
need Python validation before porting. This notebook is the Python-side
companion, run after the fact: real validation for VAD (a pre-made
ONNX, same pattern as ASR/TTS), and an honest writeup of why Denoise
doesn't get the same treatment.

- VAD: Silero VAD — pre-made ONNX, validated here on real speech.
- Denoise: RNNoise (the actual baseline wired into the Rust pipeline via
  the `nnnoiseless` crate) has no official ONNX or working Python
  binding — both tried here, both broken, documented below. DeepFilterNet3
  (CLAUDE.md's upgrade path) *does* have a legitimate pre-exported ONNX,
  but it takes custom ERB/spectral features as input rather than raw
  audio, which is its own feature-extraction project (comparable in
  scope to ASR's mel-spectrogram work) -- documented, not implemented,
  here.


## 1. VAD: Silero VAD -- download pre-made ONNX, no conversion

In [1]:
from pathlib import Path

from huggingface_hub import hf_hub_download

VAD_REPO = "onnx-community/silero-vad"
FINAL_DIR = Path("../models/vad")

onnx_path = hf_hub_download(VAD_REPO, "onnx/model.onnx")
onnx_path


/opt/homebrew/Caskroom/miniconda/base/envs/audio-lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/Users/minhld/.cache/huggingface/hub/models--onnx-community--silero-vad/snapshots/e71cae966052b992a7eca6b17738916ce0eca4ec/onnx/model.onnx'

## 2. Validate: does it actually separate speech from non-speech?

Silero VAD is stateful (512-sample chunks @ 16kHz, a small recurrent
hidden state carried between calls) -- see `models/vad/MANIFEST.md` for
the exact I/O signature this and `src/stages/vad.rs` were built against.


In [2]:
import numpy as np
import onnxruntime as ort

CHUNK = 512
SR = 16000

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])


def run_vad(audio: np.ndarray) -> np.ndarray:
    state = np.zeros((2, 1, 128), dtype=np.float32)
    sr = np.array(SR, dtype=np.int64)
    probs = []
    for start in range(0, len(audio) - CHUNK + 1, CHUNK):
        chunk = audio[start : start + CHUNK].astype(np.float32)[None, :]
        out, state = sess.run(None, {"input": chunk, "state": state, "sr": sr})
        probs.append(float(out[0, 0]))
    return np.array(probs)


In [3]:
from datasets import load_dataset

# quocanh34/viet_vivos: a Parquet mirror of VIVOS -- the official `vivos`
# HF dataset uses a loading script recent `datasets` versions no longer
# support (same note as experiments/02_asr.ipynb).
ds = load_dataset("quocanh34/viet_vivos", split="test", streaming=True)
samples = [s for _, s in zip(range(3), ds)]

for s in samples:
    audio = np.array(s["audio"]["array"], dtype=np.float32)
    probs = run_vad(audio)
    speech_frac = (probs > 0.5).mean()
    print(f"{s['transcription']!r}: {len(audio) / SR:.2f}s, mean_prob={probs.mean():.3f}, frac_speech={speech_frac:.3f}")

silence = np.zeros(SR, dtype=np.float32)
print(f"pure silence: mean_prob={run_vad(silence).mean():.3f}")

noise = (np.random.randn(SR) * 0.01).astype(np.float32)
print(f"quiet white noise: mean_prob={run_vad(noise).mean():.3f}")


'nửa vòng trái đất hơn bảy năm': 2.38s, mean_prob=0.692, frac_speech=0.703
'bốn chiếc trống đại phèng la': 2.62s, mean_prob=0.556, frac_speech=0.549
'của những thế lực hùng mạnh': 2.38s, mean_prob=0.529, frac_speech=0.541
pure silence: mean_prob=0.007
quiet white noise: mean_prob=0.010


Real speech gives mean probabilities in the ~0.5-0.7 range with most
frames above the 0.5 threshold; pure silence and quiet white noise both
sit near zero. Confirms the model correctly separates speech from
non-speech -- consistent with what `src/stages/vad.rs`'s own integration
test checks against a committed real-speech fixture (see that test's
doc comment for why a *synthetic tone* doesn't work here: Silero VAD is
a trained classifier, not an energy detector).


## 3. Package into `models/vad/`

In [4]:
import shutil
from datetime import date

FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(onnx_path, FINAL_DIR / "model.onnx")

manifest = f"""# vad

- Source: [{VAD_REPO}](https://huggingface.co/{VAD_REPO}), `onnx/model.onnx`
  (fp32) -- pre-made ONNX, not exported by us. Original model:
  [snakers4/silero-vad](https://github.com/snakers4/silero-vad).
- License: MIT
- Downloaded/repackaged: {date.today().isoformat()} (via experiments/01_denoise_vad.ipynb)
- Precision: fp32. The source repo also ships fp16/int8/uint8/q4/bnb4
  variants (same pattern as the Whisper/PhoWhisper ONNX repos) --
  quantization not yet evaluated for this stage, tracked in docs/TODO.md.
- I/O signature (used by src/stages/vad.rs):
  - `input`: `[1, 512]` f32 -- one 512-sample chunk (32ms @ 16kHz)
  - `state`: `[2, 1, 128]` f32 -- recurrent hidden state, carried between
    calls; reset to zeros at the start of each utterance
  - `sr`: scalar int64 -- sample rate (16000 here; Silero also supports
    8000 with a 256-sample chunk, not used in this pipeline)
  - `output`: `[1, 1]` f32 -- speech probability for this chunk
  - `stateN`: `[2, 1, 128]` f32 -- updated state, feed into the next call
- Validated (this notebook): mean speech probability ~0.5-0.7 on 3 real
  VIVOS utterances, ~0.007-0.009 on pure silence and quiet white noise --
  correctly separates speech from non-speech. Not yet validated against
  a labeled VAD benchmark (precise boundary accuracy), only this coarse
  separation check.
- Rust: `src/stages/vad.rs`'s `Vad::trim` chunks the input, runs each
  512-sample chunk through the model carrying `state` forward, then
  trims samples before the first and after the last chunk with
  probability > 0.5. Interior pauses are left alone. Real integration
  test against a committed WAV fixture passes (`cargo test -- --ignored`).

Files: `model.onnx`.
"""
(FINAL_DIR / "MANIFEST.md").write_text(manifest)
sorted(p.name for p in FINAL_DIR.iterdir())


['.gitkeep', 'MANIFEST.md', 'model.onnx']

## 4. Denoise: why there's no equivalent Python validation for RNNoise

Every other stage in this project (VAD above, ASR, MT, TTS) follows the
same shape: find a pre-made ONNX (or export one), validate it in
Python, then port to Rust with the exact recipe already proven. RNNoise
breaks that pattern -- it's a small RNN that predates the
ONNX-everywhere convention, and:

- **No official ONNX export exists.** RNNoise (xiph/rnnoise) ships as a
  C library with weights baked into C source, not a portable model
  format.
- **`pyrnnoise`** (the one PyPI package wrapping the real C RNNoise
  library) is broken in this environment -- its dependency chain
  (`audiolab` -> `av`) fails to import (`ModuleNotFoundError: No module
  named 'av.option'`), a version incompatibility in an unmaintained
  transitive dependency, not something fixable by pinning our own
  requirements.
- **`deepfilternet`** (the PyPI package for CLAUDE.md's *upgrade path*,
  tried as an alternative Python-side denoise baseline) is *also*
  broken here -- it imports `torchaudio.backend.common`, an API
  `torchaudio` removed in the versions available for our current
  `torch`. No compatible older `torchaudio` is available to install
  either.

Both attempts are left below, uncommented-out, as an honest record
rather than silently skipped.


In [5]:
# Attempt 1: pyrnnoise (real C RNNoise via Python bindings)
try:
    import pyrnnoise  # noqa: F401
    print("pyrnnoise import OK")
except Exception as e:
    print(f"pyrnnoise import FAILED: {type(e).__name__}: {e}")


objc[34369]: Class AVFFrameReceiver is implemented in both /opt/homebrew/Cellar/ffmpeg/8.1.2_1/lib/libavdevice.62.3.102.dylib (0x174080328) and /opt/homebrew/Caskroom/miniconda/base/envs/audio-lab/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.3.102.dylib (0x1743683a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[34369]: Class AVFAudioReceiver is implemented in both /opt/homebrew/Cellar/ffmpeg/8.1.2_1/lib/libavdevice.62.3.102.dylib (0x174080378) and /opt/homebrew/Caskroom/miniconda/base/envs/audio-lab/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.3.102.dylib (0x1743683f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


pyrnnoise import FAILED: ModuleNotFoundError: No module named 'av.option'


In [6]:
# Attempt 2: deepfilternet (DeepFilterNet3, CLAUDE.md's denoise upgrade path)
try:
    from df.enhance import enhance, init_df  # noqa: F401
    print("deepfilternet import OK")
except Exception as e:
    print(f"deepfilternet import FAILED: {type(e).__name__}: {e}")


deepfilternet import FAILED: ModuleNotFoundError: No module named 'torchaudio.backend'


### What's actually validated for RNNoise, then

Not in Python. `nnnoiseless` (the Rust crate actually wired into
`src/stages/denoise.rs`) is a from-scratch pure-Rust reimplementation of
RNNoise with the trained weights compiled directly into the crate --
there's no separate Python-side model to cross-check it against here.
Its correctness is validated by `src/stages/denoise.rs`'s own Rust unit
tests instead:
- `process_resamples_to_16khz_at_roughly_a_third_the_length` -- output
  shape/rate sanity check
- `lowpass_filter_attenuates_frequencies_above_the_new_nyquist` -- a
  20kHz tone gets attenuated by the post-denoise resample filter while a
  1kHz tone survives, i.e. the filter actually filters (frequency-domain
  correctness), not just decimates

Run with `cargo test` from the repo root.

### DeepFilterNet3: a real pre-exported ONNX exists, just not a drop-in

Unlike RNNoise, DeepFilterNet3 *does* have legitimate pre-exported ONNX
files (e.g. [soniqo/DeepFilterNet3-ONNX](https://huggingface.co/soniqo/DeepFilterNet3-ONNX),
MIT, sourced directly from the official `Rikorose/DeepFilterNet` v0.5.6
release, validated against PyTorch to ~1e-6 per its own export
manifest). But its input isn't raw audio:


In [7]:
from huggingface_hub import hf_hub_download
import onnx

dfn3_path = hf_hub_download("soniqo/DeepFilterNet3-ONNX", "deepfilter.onnx")
m = onnx.load(dfn3_path, load_external_data=False)
print("INPUTS:")
for inp in m.graph.input:
    dims = [d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(" ", inp.name, dims)
print("OUTPUTS:")
for out in m.graph.output:
    dims = [d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]
    print(" ", out.name, dims)


INPUTS:
  feat_erb [1, 1, 'frames', 32]
  feat_spec [1, 2, 'frames', 96]
OUTPUTS:
  erb_mask ['Sigmoiderb_mask_dim_0', 1, 'frames', 'Sigmoiderb_mask_dim_3']
  df_coefs ['Transposedf_coefs_dim_0', 5, 'frames', 'Transposedf_coefs_dim_3', 2]


`feat_erb`/`feat_spec` are DeepFilterNet's own ERB (equivalent
rectangular bandwidth) and STFT spectral features -- not the same as
Whisper's mel filterbank, and not computed by this graph either. Using
this model means implementing DeepFilterNet's specific feature
extraction (and, on the output side, applying the ERB mask + deep
filtering coefficients back onto the STFT to reconstruct audio) from
scratch -- a project comparable in scope to ASR's mel-spectrogram work
(`docs/decisions/mel-spec-crate-whisper-compat-bugs.md`), not a quick
add. Tracked as a real follow-up in `docs/TODO.md`, not attempted here.
